# Milestone 2.4.1 — Kaggle T4 runtime candidate

This notebook orchestrates the existing Milestone 2.3 APIs. Select Kaggle GPU T4 x2. Both devices are inventoried, but all model work is explicitly restricted to `cuda:0`. Full training and smoke tests are opt-in and disabled by default. The Kaggle runtime profile remains **candidate** until actual Kaggle probes pass.

Model/tokenizer revision `12040accade4e8a0f71eabdb258fecc2e7e948be` is locally characterized, not independently approved for production. Baseline/canonical results are a whole-system comparison; they do not isolate byte-feature effects. The held-out test split is not loaded or evaluated.

Validation begins with an incrementally persisted `validation_report.json`. It records each probe state and the context fingerprint; failure evidence is separate from scientific metrics. Atomic replacement depends on filesystem support and is not guaranteed across sudden power loss or mounted/network filesystems.

The bounded probes include FP16 autocast and GradScaler update verification on `cuda:0`; the result must match the effective precision resolved into the execution-context fingerprint.


Recovery coverage includes epoch-boundary recovery/resume from the published training snapshot. The bounded probe uses train and validation splits only and never loads the held-out test split.


In [ ]:
# Safe defaults: Run All will not start probes or training.
PROFILE_SELECTION = 'kaggle-t4-candidate'
CANDIDATE_PROFILE = None
VALIDATION_RUN_ID = None
ALLOW_PINNED_ASSET_DOWNLOAD = False
ALLOW_REPOSITORY_CLONE = False
RUN_CUDA_SMOKE_TESTS = False
RUN_FULL_BASELINE_TRAINING = False
RUN_FULL_CANONICAL_TRAINING = False
REQUESTED_MIXED_PRECISION = 'fp16'
MODEL_ID = TOKENIZER_ID = 'distilbert-base-uncased'
MODEL_REVISION = TOKENIZER_REVISION = '12040accade4e8a0f71eabdb258fecc2e7e948be'
DEVICE_NAME = 'cuda:0'
SEED, MICROBATCH_SIZE, GRADIENT_ACCUMULATION_STEPS = 42, 2, 8
EFFECTIVE_BATCH_SIZE = MICROBATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
EPOCHS, MAX_EXPANDED_LENGTH, MAX_BYTE_SPAN_LENGTH = 3, 128, 512
LEARNING_RATE, WEIGHT_DECAY, WARMUP_RATIO, GRADIENT_CLIP_NORM = 2e-5, .01, .1, 1.0
VALIDATION_INTERVAL_STEPS, NUM_WORKERS = 0, 0
print({'profile_selection':PROFILE_SELECTION,'profile_status':'candidate','device':DEVICE_NAME,'effective_batch_size':EFFECTIVE_BATCH_SIZE,'automatic_training':False})

## 1. Repository setup and validation report initialization


In [ ]:
import subprocess, sys, os, json, time, uuid, tempfile
from pathlib import Path

def setup_repository(input_repo, work_repo, repository_ref, repository_commit, *, allow_clone, run=subprocess.run):
 input_repo=Path(input_repo); work_repo=Path(work_repo)
 if (input_repo/'src'/'training.py').is_file(): repo_root=input_repo
 elif (work_repo/'src'/'training.py').is_file(): repo_root=work_repo
 elif not allow_clone: raise RuntimeError(f'Repository source not found. Attach it at {input_repo}, or explicitly set ALLOW_REPOSITORY_CLONE=True to clone into {work_repo}.')
 else:
  if work_repo.exists(): raise FileExistsError(f'Repository destination already exists but is incomplete: {work_repo}; choose a fresh path or attach a source dataset.')
  try:
   run(['git','clone','--depth','1','--branch',repository_ref,'https://github.com/Suhaniverma2004/nlp-defense.git',str(work_repo)],check=True)
   run(['git','-C',str(work_repo),'fetch','--depth','1','origin',repository_commit],check=True)
   run(['git','-C',str(work_repo),'checkout','--detach',repository_commit],check=True)
  except (OSError,subprocess.CalledProcessError) as exc: raise RuntimeError(f'Could not fetch the pinned source; attach it or enable Kaggle Internet. Details: {exc}') from exc
  repo_root=work_repo
 try: source_commit=run(['git','-C',str(repo_root),'rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
 except (OSError,subprocess.CalledProcessError):
  marker=repo_root/'MILESTONE_2_4_SOURCE_COMMIT.txt'; source_commit=marker.read_text(encoding='utf-8').strip() if marker.is_file() else None
 if source_commit!=repository_commit: raise RuntimeError(f'Repository source must identify inspected commit {repository_commit}; found {source_commit!r}.')
 return repo_root,source_commit

SESSION_ID=time.strftime('%Y%m%d-%H%M%S')+'-'+uuid.uuid4().hex[:8]
ARTIFACT_ROOT=Path('/kaggle/working')/f'nlp-defense-m24-{SESSION_ID}'; SPLIT_DIR=Path('/kaggle/working')/f'nlp-defense-m24-splits-{SESSION_ID}'
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True); REPORT_PATH=ARTIFACT_ROOT/'validation_report.json'
initial={'schema':1,'report_status':'repository_setup','persistence_note':'Atomic replacement is filesystem-dependent; it cannot guarantee sudden-power-loss or mounted/network-storage durability.','preliminary_context':{'profile_selection':PROFILE_SELECTION},'runs':[]}
fd,tmp=tempfile.mkstemp(prefix='.validation_report.',suffix='.tmp',dir=ARTIFACT_ROOT)
with os.fdopen(fd,'w',encoding='utf-8') as stream: json.dump(initial,stream,sort_keys=True,indent=2); stream.flush(); os.fsync(stream.fileno())
os.replace(tmp,REPORT_PATH)
def export_current_validation_report():
 import shutil
 snapshot=json.loads(REPORT_PATH.read_text(encoding='utf-8'))
 if 'runtime_report' in globals():
  runtime_report['validation_report']=snapshot; runtime_report['validation_report_status']=snapshot.get('report_status','unknown')
  runtime_report['validation_context_fingerprint']=snapshot.get('runs',[])[-1].get('context_fingerprint') if snapshot.get('runs') else None
  fd,runtime_tmp=tempfile.mkstemp(prefix='.runtime_report.',suffix='.tmp',dir=ARTIFACT_ROOT)
  with os.fdopen(fd,'w',encoding='utf-8') as stream: json.dump(runtime_report,stream,indent=2,default=str); stream.flush(); os.fsync(stream.fileno())
  os.replace(runtime_tmp,ARTIFACT_ROOT/'runtime_report.json')
 archive=Path('/kaggle/working')/f'nlp-defense-m24-validation-{SESSION_ID}-{uuid.uuid4().hex[:8]}'
 shutil.make_archive(str(archive),'zip',root_dir=ARTIFACT_ROOT); return str(archive)

INPUT_REPO=Path('/kaggle/input/nlp-defense-source'); WORK_REPO=Path('/kaggle/working/nlp-defense')
REPOSITORY_REF='main'; REPOSITORY_COMMIT='65391a2b0a715bdb67156df4aa1d5725c1edbc76'
REPO_ROOT,source_commit=setup_repository(INPUT_REPO,WORK_REPO,REPOSITORY_REF,REPOSITORY_COMMIT,allow_clone=ALLOW_REPOSITORY_CLONE)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
from src.kaggle_runtime import initialize_probe_report
if not REPORT_PATH.exists(): initialize_probe_report(REPORT_PATH,preliminary_context={'profile_selection':PROFILE_SELECTION,'repository_commit':source_commit})
print('Repository:',REPO_ROOT,'commit:',source_commit,'report:',REPORT_PATH)

## 2. Environment and GPU detection
Inventory both T4s; select `cuda:0` only.


In [ ]:
import json, os, platform, subprocess, sys
from pathlib import Path
import torch
assert sys.version_info >= (3,10), f'Python 3.10+ required, found {platform.python_version()}'
if not torch.cuda.is_available():
 from src.kaggle_runtime import record_probe_failure
 error=RuntimeError('CUDA unavailable: select GPU T4 x2 in Kaggle settings and restart.')
 record_probe_failure(REPORT_PATH,'cuda_tensor_probe',{'python':platform.python_version(),'torch':torch.__version__},error)
 try: export_current_validation_report()
 except Exception as export_error: error.add_note(f'Could not export failure snapshot: {export_error!r}')
 raise error
gpus=[]
for i in range(torch.cuda.device_count()):
 p=torch.cuda.get_device_properties(i); gpus.append({'index':i,'name':p.name,'memory_bytes':int(p.total_memory),'memory_gib':round(p.total_memory/2**30,3),'capability':[p.major,p.minor]})
if not gpus: raise RuntimeError('No CUDA devices found.')
device=torch.device('cuda:0'); torch.cuda.set_device(device)
try:
 rows=subprocess.run(['nvidia-smi','--query-gpu=driver_version,index,name,memory.total,compute_cap','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.splitlines()
 drivers=sorted({row.split(',',1)[0].strip() for row in rows if row.strip()})
 if len(rows)!=torch.cuda.device_count() or len(drivers)!=1: raise RuntimeError('nvidia-smi inventory does not match CUDA device inventory.')
 driver_version=drivers[0]
except (OSError,subprocess.CalledProcessError,RuntimeError) as exc:
 from src.kaggle_runtime import record_probe_failure
 record_probe_failure(REPORT_PATH,'runtime_profile_validation',{'gpus':gpus},exc)
 try: export_current_validation_report()
 except Exception as export_error: exc.add_note(f'Could not export failure snapshot: {export_error!r}')
 raise
runtime_report={'profile_selection':PROFILE_SELECTION,'profile_status':'candidate','python':platform.python_version(),'torch':torch.__version__,'torch_cuda_runtime':torch.version.cuda,'nvidia_driver_version':driver_version,'cuda_device_count':torch.cuda.device_count(),'gpus':gpus,'nvidia_smi_inventory':rows,'selected_device':str(device),'selected_name':torch.cuda.get_device_name(0),'selected_capability':list(torch.cuda.get_device_capability(0)),'selected_memory_bytes':int(torch.cuda.get_device_properties(0).total_memory),'automatic_multi_gpu':False}
print(json.dumps(runtime_report,indent=2)); assert str(device)=='cuda:0' 

## 3. Candidate dependency compatibility
Fail closed on exact Kaggle package versions; never reinstall PyTorch.


In [ ]:
from importlib.metadata import distributions
import numpy,pandas,sklearn,transformers,tokenizers
from src.kaggle_runtime import EXPECTED_VERSIONS,KAGGLE_PROFILE_ID,validate_candidate_runtime,record_probe_failure
observed_versions={'python':platform.python_version(),'torch':torch.__version__,'torch_cuda_runtime':str(torch.version.cuda),'transformers':transformers.__version__,'tokenizers':tokenizers.__version__,'numpy':numpy.__version__,'pandas':pandas.__version__,'scikit-learn':sklearn.__version__}
package_inventory=sorted(f"{d.metadata.get('Name')}=={d.version}" for d in distributions() if d.metadata.get('Name'))
try: CANDIDATE_PROFILE=validate_candidate_runtime(observed_versions,gpus,str(device),driver_version,PROFILE_SELECTION)
except Exception as exc:
 record_probe_failure(REPORT_PATH,'runtime_profile_validation',{'versions':observed_versions,'package_inventory':package_inventory,'gpus':gpus,'driver':driver_version},exc)
 try: export_current_validation_report()
 except Exception as export_error: exc.add_note(f'Could not export failure snapshot: {export_error!r}')
 raise
runtime_report.update({'profile_id':KAGGLE_PROFILE_ID,'versions':observed_versions,'expected_versions':EXPECTED_VERSIONS,'resolved_package_inventory':package_inventory,'runtime_profile':CANDIDATE_PROFILE})
print({'profile_id':KAGGLE_PROFILE_ID,'status':'candidate','versions':observed_versions,'package_count':len(package_inventory),'selected_gpu':torch.cuda.get_device_name(0)})

## 4. Dataset provenance and split verification
Only train and validation rows are loaded; held-out rows remain untouched.


In [ ]:
from src.sst2_data import prepare_reproducible_splits, load_split
def first_existing(paths):
 for p in paths:
  p = Path(p)
  if p.is_file(): return p
 return None
TRAIN_SOURCE = first_existing([REPO_ROOT/'data/sst2/sst2_train.csv','/kaggle/input/sst2/sst2_train.csv','/kaggle/input/glue-sst2/train.csv'])
VALIDATION_SOURCE = first_existing([REPO_ROOT/'data/sst2/sst2_validation.csv','/kaggle/input/sst2/sst2_validation.csv','/kaggle/input/glue-sst2/validation.csv'])
if TRAIN_SOURCE is None or VALIDATION_SOURCE is None: raise FileNotFoundError('Attach local SST-2 train and original-validation CSVs; no data download is attempted.')
SPLIT_DIR.mkdir(parents=True,exist_ok=True)
if not (SPLIT_DIR/'manifest.json').exists():
 if any(SPLIT_DIR.iterdir()): raise FileExistsError(f'Non-empty split path without manifest: {SPLIT_DIR}')
 prepare_reproducible_splits(TRAIN_SOURCE,VALIDATION_SOURCE,SPLIT_DIR,seed=SEED)
train_frame, manifest = load_split(SPLIT_DIR,'train'); validation_frame, validation_manifest = load_split(SPLIT_DIR,'validation')
if manifest != validation_manifest or set(train_frame.sentence)&set(validation_frame.sentence): raise ValueError('Split provenance mismatch or train/validation text overlap.')
print(json.dumps({'train_rows':len(train_frame),'validation_rows':len(validation_frame),'train_source_sha256':manifest['source']['train_sha256'],'original_validation_sha256':manifest['source']['original_validation_sha256'],'seed':manifest['seed'],'strategy':manifest['split_strategy'],'held_out_split_loaded':False},indent=2))

## 5. Immutable model and tokenizer revision verification
The pinned revision is tested locally, not approved as a production revision.


In [ ]:
from transformers import AutoConfig,AutoTokenizer
from huggingface_hub import snapshot_download
from src.training import PINNED_REPRODUCIBILITY_REVISION
from src.kaggle_runtime import record_probe_failure

def verify_pinned_hf_assets(model_id,model_revision,tokenizer_id,tokenizer_revision,*,pinned_revision,local_files_only,snapshot_download_fn):
 import hashlib
 from pathlib import Path
 expected_repo='distilbert-base-uncased'
 if model_id!=expected_repo or tokenizer_id!=expected_repo:
  raise RuntimeError(f'Model and tokenizer repository IDs must both be {expected_repo!r}; got {model_id!r} and {tokenizer_id!r}.')
 if model_id!=tokenizer_id or model_revision!=tokenizer_revision:
  raise RuntimeError('Model and tokenizer repository IDs/revisions must be identical.')
 if not isinstance(model_revision,str) or len(model_revision)!=40 or any(c not in '0123456789abcdef' for c in model_revision):
  raise RuntimeError(f'Revision must be an immutable lowercase 40-character commit SHA; got {model_revision!r}.')
 if model_revision!=pinned_revision or tokenizer_revision!=pinned_revision:
  raise RuntimeError(f'Model and tokenizer must use the pinned revision {pinned_revision!r}.')
 required_files=('config.json','tokenizer.json','tokenizer_config.json','vocab.txt')
 requested_files=[*required_files,'special_tokens_map.json','added_tokens.json','model.safetensors','pytorch_model.bin']
 def resolve_snapshot(repo_id,revision,role):
  try: snapshot=Path(snapshot_download_fn(repo_id=repo_id,revision=revision,local_files_only=local_files_only,allow_patterns=requested_files)).resolve(strict=True)
  except Exception as exc: raise RuntimeError(f'Could not resolve {role} snapshot for {repo_id}@{revision} with local_files_only={local_files_only}: {exc}') from exc
  cache_repo='models--'+repo_id.replace('/','--')
  if not snapshot.is_dir() or snapshot.name!=revision or snapshot.parent.name!='snapshots' or snapshot.parent.parent.name!=cache_repo:
   raise RuntimeError(f'{role} resolved to an unexpected repository snapshot identity: {snapshot}')
  cache_root=snapshot.parent.parent.resolve(strict=True)
  missing=[name for name in required_files if not (snapshot/name).is_file()]
  if missing: raise RuntimeError(f'{role} snapshot {cache_repo}/snapshots/{revision} is missing required assets: {missing}')
  weights=[name for name in ('model.safetensors','pytorch_model.bin') if (snapshot/name).is_file()]
  if not weights: raise RuntimeError(f'{role} snapshot {cache_repo}/snapshots/{revision} has no supported model weights (model.safetensors or pytorch_model.bin).')
  hashes={}
  for name in (*required_files,*weights,'special_tokens_map.json','added_tokens.json'):
   asset=snapshot/name
   if not asset.is_file(): continue
   resolved=asset.resolve(strict=True)
   try: resolved.relative_to(cache_root)
   except ValueError as exc: raise RuntimeError(f'{role} asset resolves outside its Hugging Face repository cache: {name}') from exc
   hashes[name]=hashlib.sha256(resolved.read_bytes()).hexdigest()
  return snapshot,cache_repo,hashes,weights
 model_snapshot,model_cache,model_hashes,weights=resolve_snapshot(model_id,model_revision,'model')
 tokenizer_snapshot,tokenizer_cache,tokenizer_hashes,_=resolve_snapshot(tokenizer_id,tokenizer_revision,'tokenizer')
 if model_snapshot!=tokenizer_snapshot or model_cache!=tokenizer_cache or model_hashes!=tokenizer_hashes:
  raise RuntimeError('Model and tokenizer did not resolve to the same repository snapshot and asset contents.')
 return {'model_id':model_id,'model_revision':model_snapshot.name,'tokenizer_id':tokenizer_id,'tokenizer_revision':tokenizer_snapshot.name,'model_snapshot_identity':f'{model_cache}/snapshots/{model_snapshot.name}','tokenizer_snapshot_identity':f'{tokenizer_cache}/snapshots/{tokenizer_snapshot.name}','asset_sha256':model_hashes,'required_assets':list(required_files),'weight_assets':weights}

try:
 assert MODEL_REVISION==TOKENIZER_REVISION==PINNED_REPRODUCIBILITY_REVISION
 local_only=not ALLOW_PINNED_ASSET_DOWNLOAD
 asset_verification=verify_pinned_hf_assets(MODEL_ID,MODEL_REVISION,TOKENIZER_ID,TOKENIZER_REVISION,pinned_revision=PINNED_REPRODUCIBILITY_REVISION,local_files_only=local_only,snapshot_download_fn=snapshot_download)
 tok=AutoTokenizer.from_pretrained(TOKENIZER_ID,revision=TOKENIZER_REVISION,use_fast=True,local_files_only=local_only)
 model_config=AutoConfig.from_pretrained(MODEL_ID,revision=MODEL_REVISION,local_files_only=local_only)
 if not tok.is_fast: raise RuntimeError('Fast tokenizer with offset support required.')
except Exception as exc:
 record_probe_failure(REPORT_PATH,'tokenizer_revision_verified',{'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'tokenizer_id':TOKENIZER_ID,'tokenizer_revision':TOKENIZER_REVISION},exc)
 try: export_current_validation_report()
 except Exception as export_error: exc.add_note(f'Could not export failure snapshot: {export_error!r}')
 raise
runtime_report.update({'model_id':MODEL_ID,'model_revision':asset_verification['model_revision'],'tokenizer_id':TOKENIZER_ID,'tokenizer_revision':asset_verification['tokenizer_revision'],'fast_tokenizer':True,'verified_hf_assets':asset_verification})
print({'requested_revision':MODEL_REVISION,'resolved_model_snapshot':asset_verification['model_snapshot_identity'],'resolved_tokenizer_snapshot':asset_verification['tokenizer_snapshot_identity'],'verified_asset_names':sorted(asset_verification['asset_sha256']),'fast_tokenizer':True,'download_allowed':ALLOW_PINNED_ASSET_DOWNLOAD})
del tok,model_config

## 6. Precision resolution and shared settings
Resolve precision once; smoke validation checks all cached settings before execution.


In [ ]:
from src.training import TrainingConfig,load_pretrained_components,resolve_mixed_precision,execution_metadata
from src.kaggle_runtime import record_probe_failure
try: effective_dtype=resolve_mixed_precision(REQUESTED_MIXED_PRECISION,device)
except Exception as exc:
 record_probe_failure(REPORT_PATH,'fp16_autocast_grad_scaler_update',{'requested_precision':REQUESTED_MIXED_PRECISION,'device':DEVICE_NAME},exc)
 try: export_current_validation_report()
 except Exception as export_error: exc.add_note(f'Could not export failure snapshot: {export_error!r}')
 raise
effective_precision='none' if effective_dtype is None else str(effective_dtype).replace('torch.','')
SHARED=dict(model_id=MODEL_ID,model_revision=MODEL_REVISION,tokenizer_id=TOKENIZER_ID,tokenizer_revision=TOKENIZER_REVISION,max_expanded_length=MAX_EXPANDED_LENGTH,max_byte_span_length=MAX_BYTE_SPAN_LENGTH,batch_size=MICROBATCH_SIZE,gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,seed=SEED,epochs=EPOCHS,learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,warmup_ratio=WARMUP_RATIO,gradient_clip_norm=GRADIENT_CLIP_NORM,mixed_precision=REQUESTED_MIXED_PRECISION,validation_interval_steps=VALIDATION_INTERVAL_STEPS,num_workers=NUM_WORKERS,device=DEVICE_NAME)
runtime_report.update({'effective_precision':effective_precision,'precision_execution_metadata':execution_metadata(device,effective_dtype),'training_config_shared':SHARED,'comparison_scope':'whole-system; byte-feature effect is not isolated'})
print({'requested_precision':REQUESTED_MIXED_PRECISION,'effective_precision':effective_precision,'execution':execution_metadata(device,effective_dtype),'shared_settings':SHARED,'comparison':'whole-system only; byte-feature effect is not isolated'})

## 7. Bounded CUDA smoke, checkpoint reload, and epoch-boundary resume
Disabled by default. When explicitly enabled, both models use at most four training and four validation examples. Existing trainer/checkpoint APIs are used. The cell simulates a pointer-write interruption after snapshot publication, invokes snapshot recovery, resumes, reloads the best checkpoint offline, and runs inference.

In [ ]:
from contextlib import contextmanager
from unittest.mock import patch
from src.alignment import align_text
from src.kaggle_runtime import (REQUIRED_PROBES,complete_probe_run,require_full_training_gate,run_reported_probe,start_probe_run,validate_candidate_runtime,validation_context_fingerprint,record_probe_failure,build_authoritative_execution_context,build_smoke_training_config)
from src.checkpointing import load_checkpoint,recover_latest_resume_checkpoint,validate_checkpoint_bundle

def verify_source_partition(text,tokenizer):
 result=align_text(text,tokenizer,max_expanded_length=MAX_EXPANDED_LENGTH,max_bytes_per_span=MAX_BYTE_SPAN_LENGTH)
 if not result.coverage.fully_represented: raise AssertionError(f'Input not fully represented: {text!r}')
 cursor=0; pieces=[]; raw_bytes=[]
 for span in result.source_spans:
  if span.start!=cursor or text[span.start:span.end]!=span.text: raise AssertionError('Token/gap spans do not exactly partition source order.')
  cursor=span.end; pieces.append(span.text); raw_bytes.extend(value-1 for value in span.byte_encoding.ids)
 if cursor!=len(text) or ''.join(pieces)!=text or bytes(raw_bytes)!=text.encode('utf-8',errors='strict'): raise AssertionError('Source/UTF-8 reconstruction failed.')
 for unit in result.units:
  if unit.kind=='special' and (unit.source_start!=unit.source_end or unit.byte_length!=0): raise AssertionError('Structural special token owns source bytes.')
 return result

def source_tree_digest(repository_root):
 import hashlib
 root=Path(repository_root).resolve(strict=True)
 candidates=[]
 for folder in ('src','experiment'):
  directory=root/folder
  if not directory.is_dir(): raise FileNotFoundError(f'Required source directory missing in checked-out repository: {directory}')
  candidates.extend(sorted(directory.rglob('*.py')))
 for relative in ('requirements.txt','notebooks/milestone_2_4_kaggle_gpu.ipynb'):
  path=root/relative
  if not path.is_file(): raise FileNotFoundError(f'Required source-tree fingerprint input missing: {path}')
  candidates.append(path)
 digest=hashlib.sha256()
 for candidate in sorted(set(candidates),key=lambda item:item.relative_to(root).as_posix()):
  resolved=candidate.resolve(strict=True)
  try: relative=resolved.relative_to(root).as_posix()
  except ValueError as exc: raise RuntimeError(f'Source fingerprint input escapes checked-out repository root: {candidate}') from exc
  digest.update(relative.encode('utf-8')); digest.update(b'\0'); digest.update(resolved.read_bytes()); digest.update(b'\0')
 return digest.hexdigest()

def current_validation_context():
 import numpy,pandas,sklearn,transformers,tokenizers,torch
 from dataclasses import asdict
 from importlib.metadata import distributions
 live_versions={'python':platform.python_version(),'torch':torch.__version__,'torch_cuda_runtime':str(torch.version.cuda),'transformers':transformers.__version__,'tokenizers':tokenizers.__version__,'numpy':numpy.__version__,'pandas':pandas.__version__,'scikit-learn':sklearn.__version__}
 inventory=sorted(f"{d.metadata.get('Name')}=={d.version}" for d in distributions() if d.metadata.get('Name'))
 g=[]
 for i in range(torch.cuda.device_count()):
  q=torch.cuda.get_device_properties(i); g.append({'index':i,'name':q.name,'memory_bytes':int(q.total_memory),'capability':[q.major,q.minor]})
 rows=subprocess.run(['nvidia-smi','--query-gpu=driver_version,index,name,memory.total,compute_cap','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.splitlines()
 drivers=sorted({line.split(',',1)[0].strip() for line in rows if line.strip()})
 if not torch.cuda.is_available() or DEVICE_NAME!='cuda:0' or len(drivers)!=1: raise RuntimeError('Current CUDA/device/driver state is incompatible with the candidate profile.')
 if not isinstance(device,torch.device) or device!=torch.device('cuda:0') or torch.cuda.current_device()!=0: raise RuntimeError('Cached device object and active CUDA device must both be cuda:0.')
 profile=validate_candidate_runtime(live_versions,g,DEVICE_NAME,drivers[0],PROFILE_SELECTION)
 from src.training import PINNED_REPRODUCIBILITY_REVISION
 if MODEL_REVISION!=TOKENIZER_REVISION or MODEL_REVISION!=PINNED_REPRODUCIBILITY_REVISION: raise RuntimeError('Model/tokenizer revision changed or is not the pinned immutable revision.')
 fresh_asset_verification=verify_pinned_hf_assets(MODEL_ID,MODEL_REVISION,TOKENIZER_ID,TOKENIZER_REVISION,pinned_revision=PINNED_REPRODUCIBILITY_REVISION,local_files_only=not ALLOW_PINNED_ASSET_DOWNLOAD,snapshot_download_fn=snapshot_download)
 if fresh_asset_verification!=asset_verification: raise RuntimeError('Resolved model/tokenizer snapshot identity or asset hashes changed since initial verification.')
 commit=subprocess.run(['git','rev-parse','HEAD'],cwd=REPO_ROOT,check=True,capture_output=True,text=True).stdout.strip()
 if commit!=REPOSITORY_COMMIT or commit!=source_commit: raise RuntimeError('Repository commit changed after validation setup.')
 fresh_manifest=json.loads((SPLIT_DIR/'manifest.json').read_text(encoding='utf-8'))
 _,train_manifest=load_split(SPLIT_DIR,'train'); _,valid_manifest=load_split(SPLIT_DIR,'validation')
 if fresh_manifest!=manifest or train_manifest!=fresh_manifest or valid_manifest!=fresh_manifest: raise RuntimeError('Dataset/source/split hashes changed after validation.')
 dtype=resolve_mixed_precision(REQUESTED_MIXED_PRECISION,torch.device(DEVICE_NAME)); precision='none' if dtype is None else str(dtype).replace('torch.','')
 authoritative_shared={'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'tokenizer_id':TOKENIZER_ID,'tokenizer_revision':TOKENIZER_REVISION,'max_expanded_length':MAX_EXPANDED_LENGTH,'max_byte_span_length':MAX_BYTE_SPAN_LENGTH,'batch_size':MICROBATCH_SIZE,'gradient_accumulation_steps':GRADIENT_ACCUMULATION_STEPS,'seed':SEED,'epochs':EPOCHS,'learning_rate':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,'warmup_ratio':WARMUP_RATIO,'gradient_clip_norm':GRADIENT_CLIP_NORM,'mixed_precision':REQUESTED_MIXED_PRECISION,'validation_interval_steps':VALIDATION_INTERVAL_STEPS,'num_workers':NUM_WORKERS,'device':DEVICE_NAME}
 overrides={'epochs':2,'examples_per_split':4}
 bindings=build_authoritative_execution_context(authoritative_shared_config=authoritative_shared,cached_shared_config=SHARED,effective_precision=precision,cached_effective_precision=effective_precision,cached_dtype=effective_dtype,requested_device=DEVICE_NAME,cached_device=device,active_cuda_index=torch.cuda.current_device(),smoke_overrides=overrides)
 from src.training import TrainingConfig
 base=TrainingConfig(**bindings['shared_training_config']); training_config=asdict(base); defaults=TrainingConfig()
 training_config['effective_precision']=precision; training_config['smoke_overrides']={'epochs':2,'examples_per_split':4}
 training_config.update({'effective_batch_size':MICROBATCH_SIZE*GRADIENT_ACCUMULATION_STEPS,'optimizer':{'name':'AdamW','betas':[0.9,0.999],'eps':1e-8,'weight_decay':WEIGHT_DECAY,'amsgrad':False},'scheduler':{'name':'LambdaLR','warmup_ratio':WARMUP_RATIO,'step_per_optimizer_update':True},'architecture_defaults':{'byte_embedding_dim':defaults.byte_embedding_dim,'byte_num_filters':defaults.byte_num_filters,'byte_kernel_sizes':list(defaults.byte_kernel_sizes),'byte_output_dim':defaults.byte_output_dim,'dropout':defaults.dropout}})
 source_digest_value=source_tree_digest(REPO_ROOT)
 context={'source_tree_sha256':source_digest_value,'runtime_profile':profile,'installed_package_inventory':inventory,'device_metadata':{'selected_device':DEVICE_NAME,'active_cuda_index':torch.cuda.current_device(),'driver_version':drivers[0],'cuda_runtime':str(torch.version.cuda),'gpus':g,'nvidia_smi_inventory':rows},'repository_commit':commit,'model':{'id':MODEL_ID,'revision':MODEL_REVISION},'tokenizer':{'id':TOKENIZER_ID,'revision':TOKENIZER_REVISION},'verified_hf_assets':fresh_asset_verification,'dataset_manifest':fresh_manifest,'training_configuration':training_config,'execution_bindings':bindings}
 execution={'bindings':bindings,'device':torch.device(bindings['device']),'effective_precision':bindings['effective_precision'],'autocast_dtype':dtype}
 return context,profile,execution

def persist_probe_report_snapshot(): return export_current_validation_report()

def require_current_candidate_gate():
 if not VALIDATION_RUN_ID: raise RuntimeError('Full training is blocked until same-session CUDA probes pass.')
 context,profile,_=current_validation_context()
 require_full_training_gate(profile,REPORT_PATH,VALIDATION_RUN_ID,context)

@contextmanager
def temporary_hf_offline():
 with patch.dict(os.environ,{'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1'}): yield

def verify_fp16_scaler(execution):
 if execution['autocast_dtype']!=torch.float16 or execution['effective_precision']!='float16': raise RuntimeError('Candidate T4 scaler probe requires context-bound FP16.')
 probe_device=execution['device']; torch.manual_seed(SEED); model=torch.nn.Linear(8,2).to(probe_device); optimizer=torch.optim.SGD(model.parameters(),lr=.01); scaler=torch.amp.GradScaler('cuda',enabled=True)
 before=[p.detach().clone() for p in model.parameters()]; x=torch.arange(32,device=probe_device,dtype=torch.float32).reshape(4,8)/32; y=torch.tensor([0,1,0,1],device=probe_device)
 with torch.autocast(device_type='cuda',dtype=execution['autocast_dtype']): loss=torch.nn.functional.cross_entropy(model(x),y)
 if not torch.isfinite(loss): raise FloatingPointError('FP16 probe loss is not finite.')
 scaler.scale(loss).backward(); scaler.unscale_(optimizer)
 if any(p.grad is None or not torch.isfinite(p.grad).all().item() for p in model.parameters()): raise FloatingPointError('FP16 probe gradients are invalid.')
 scaler.step(optimizer); scaler.update()
 if not any(not torch.equal(a,b.detach()) for a,b in zip(before,model.parameters())) or not scaler.is_enabled(): raise AssertionError('GradScaler probe did not update parameters.')
 torch.cuda.synchronize(probe_device)

def run_model_probe(kind,smoke_train,smoke_valid,provenance,execution):
 from src.training import fit_model,ClassificationCollator
 if kind not in {'baseline','canonical'}: raise ValueError(f'Unsupported model kind: {kind}')
 out=ARTIFACT_ROOT/'smoke'/VALIDATION_RUN_ID/kind; out.mkdir(parents=True,exist_ok=True)
 if any(out.iterdir()): raise FileExistsError(f'Use a fresh smoke directory: {out}')
 smoke_config=build_smoke_training_config(execution['bindings'],out)
 cfg=TrainingConfig(**smoke_config)
 expected_config=asdict(TrainingConfig(**execution['bindings']['shared_training_config']))
 expected_config['epochs']=execution['bindings']['smoke_overrides']['epochs']; expected_config['output_dir']=str(out)
 if asdict(cfg)!=expected_config: raise RuntimeError(f'{kind} TrainingConfig differs from its validated execution context.')
 if not isinstance(execution['device'],torch.device) or execution['device']!=torch.device('cuda:0') or torch.cuda.current_device()!=0: raise RuntimeError(f'{kind} smoke training must execute on the fingerprinted active cuda:0.')
 model,tokenizer=load_pretrained_components(cfg,model_kind=kind,local_files_only=not ALLOW_PINNED_ASSET_DOWNLOAD)
 collator=ClassificationCollator(tokenizer,cfg,kind); before=next(model.parameters()).detach().cpu().clone(); torch.cuda.reset_peak_memory_stats(execution['device']); started=time.perf_counter()
 with patch('src.checkpointing.write_latest_resume_pointer',side_effect=RuntimeError('simulated interruption')):
  try: fit_model(model,tokenizer,smoke_train,smoke_valid,cfg,model_kind=kind,checkpoint_root=out/'checkpoints',dataset_provenance=provenance,history_path=out/'training_history.json')
  except RuntimeError as exc:
   if 'simulated interruption' not in str(exc): raise
 snapshot=recover_latest_resume_checkpoint(out/'checkpoints')
 if snapshot is None: raise RuntimeError(f'No valid recovered snapshot for {kind}.')
 validate_checkpoint_bundle(snapshot,require_training_state=True)
 del model,tokenizer; torch.cuda.empty_cache()
 model,tokenizer=load_pretrained_components(cfg,model_kind=kind,local_files_only=True)
 result=fit_model(model,tokenizer,smoke_train,smoke_valid,cfg,model_kind=kind,checkpoint_root=out/'checkpoints',dataset_provenance=provenance,resume_checkpoint_dir=snapshot,history_path=out/'training_history.json')
 if result.device!=str(execution['device']) or result.effective_mixed_precision!=execution['effective_precision']: raise RuntimeError(f'{kind} training did not match its fingerprinted device/precision.')
 if not result.best_checkpoint_dir or not Path(result.best_checkpoint_dir).is_dir() or torch.equal(before,next(result.model.parameters()).detach().cpu()): raise AssertionError('Optimizer update or best checkpoint missing.')
 validate_checkpoint_bundle(result.best_checkpoint_dir)
 with temporary_hf_offline(): reloaded,rtok,meta=load_checkpoint(result.best_checkpoint_dir,expected_model_kind=kind,expected_model_id=MODEL_ID,expected_model_revision=MODEL_REVISION,expected_tokenizer_id=TOKENIZER_ID,expected_tokenizer_revision=TOKENIZER_REVISION,device=execution['device'])
 row=smoke_valid.iloc[0]; batch=collator([(row.sentence,int(row.label),0)]); inp=batch['inputs']; logits=reloaded(inp.to(execution['device'])).logits if kind=='canonical' else reloaded(inp['input_ids'].to(execution['device']),inp['attention_mask'].to(execution['device'])).logits
 if logits.shape!=(1,2) or not torch.isfinite(logits).all(): raise FloatingPointError('Reloaded inference invalid.')
 if any(not torch.isfinite(torch.tensor(float(v))) for v in result.best_validation_metrics.values() if isinstance(v,(int,float))): raise FloatingPointError('Non-finite validation metrics.')
 record={'optimizer_update':True,'checkpoint_integrity':True,'offline_reload_inference':True,'resume_recovery':True,'validation':result.best_validation_metrics,'elapsed_seconds':time.perf_counter()-started,'peak_cuda_bytes':torch.cuda.max_memory_allocated(execution['device']),'effective_precision':result.effective_mixed_precision,'checkpoint_kind':meta['model_kind'],'logits_shape':list(logits.shape)}
 del model,tokenizer,result,reloaded,rtok,logits; torch.cuda.empty_cache(); return record

if RUN_CUDA_SMOKE_TESTS:
 try: context,profile,execution=current_validation_context()
 except Exception as exc:
  record_probe_failure(REPORT_PATH,'runtime_profile_validation',{'versions':observed_versions,'package_inventory':package_inventory,'repository_commit':source_commit,'dataset_manifest':manifest,'model_revision':MODEL_REVISION,'tokenizer_revision':TOKENIZER_REVISION,'training_config':SHARED,'device':str(device),'active_cuda_index':torch.cuda.current_device()},exc)
  try: persist_probe_report_snapshot()
  except Exception as export_error: exc.add_note(f'Could not export failure report snapshot: {export_error!r}')
  raise
 VALIDATION_RUN_ID=start_probe_run(REPORT_PATH,context)
 runtime_report['validation_context_fingerprint']=validation_context_fingerprint(context); runtime_report['validation_run_id']=VALIDATION_RUN_ID
 def run_probe(name,fn):
  try: return run_reported_probe(REPORT_PATH,VALIDATION_RUN_ID,name,fn)
  except BaseException as exc:
   try: persist_probe_report_snapshot()
   except BaseException as export_error:
    if hasattr(exc,'add_note'): exc.add_note(f'Could not export failure report snapshot: {export_error!r}')
   raise
 run_probe('runtime_profile_validation',lambda:validate_candidate_runtime(context['runtime_profile']['versions'],context['runtime_profile']['gpus'],DEVICE_NAME,context['device_metadata']['driver_version'],PROFILE_SELECTION))
 def cuda_probe():
  x=torch.ones((32,32),device=execution['device']); y=x@x
  if y.device!=execution['device'] or not torch.isfinite(y).all().item(): raise RuntimeError('CUDA tensor probe failed on cuda:0.')
 run_probe('cuda_tensor_probe',cuda_probe)
 from transformers import AutoTokenizer
 def load_probe_tokenizer():
  fresh=verify_pinned_hf_assets(MODEL_ID,MODEL_REVISION,TOKENIZER_ID,TOKENIZER_REVISION,pinned_revision=PINNED_REPRODUCIBILITY_REVISION,local_files_only=not ALLOW_PINNED_ASSET_DOWNLOAD,snapshot_download_fn=snapshot_download)
  if fresh!=asset_verification: raise RuntimeError('Tokenizer probe resolved a different pinned snapshot or asset manifest.')
  tok=AutoTokenizer.from_pretrained(TOKENIZER_ID,revision=TOKENIZER_REVISION,use_fast=True,local_files_only=not ALLOW_PINNED_ASSET_DOWNLOAD)
  if not tok.is_fast: raise RuntimeError('Pinned tokenizer did not load as a fast tokenizer.')
  return tok
 probe_tokenizer=run_probe('tokenizer_revision_verified',load_probe_tokenizer)
 run_probe('unicode_emoji_alignment',lambda:verify_source_partition('Cafe\u0301 😀 👩\u200d💻️',probe_tokenizer))
 def whitespace_probe():
  text='  good  text \t'; r=verify_source_partition(text,probe_tokenizer); gaps=[z for z in r.gap_spans if z.text.isspace()]
  if not gaps or not any(z.start==0 for z in gaps) or not any(z.end==len(text) for z in gaps) or not any(0<z.start and z.end<len(text) for z in gaps): raise AssertionError('Whitespace gap coverage incomplete.')
 run_probe('whitespace_gap_alignment',whitespace_probe)
 def special_probe():
  r=verify_source_partition('literal [CLS] content',probe_tokenizer)
  if '[CLS]' not in ''.join(z.text for z in r.token_spans) or not any(u.kind=='special' for u in r.units): raise AssertionError('Literal and structural special-token handling failed.')
 run_probe('special_token_offset_handling',special_probe)
 run_probe('fp16_autocast_grad_scaler_update',lambda:verify_fp16_scaler(execution))
 del probe_tokenizer
 import pandas as pd
 def select_smoke_splits():
  train=train_frame.groupby('label',group_keys=False).head(2).head(4).reset_index(drop=True); valid=validation_frame.groupby('label',group_keys=False).head(2).head(4).reset_index(drop=True)
  if len(train)!=4 or len(valid)!=4 or set(train.sentence)&set(valid.sentence): raise ValueError('Need four disjoint rows in each smoke split.')
  return train,valid,{'kind':'bounded-smoke-subset','source_manifest':manifest,'train_ids':train.get('source_row_index',pd.Series(range(4))).tolist(),'validation_ids':valid.get('source_row_index',pd.Series(range(4))).tolist()}
 smoke_train,smoke_valid,provenance=run_probe('smoke_split_validation',select_smoke_splits)
 smoke_results={}
 smoke_results['baseline']=run_probe('baseline_cuda_training_checkpoint_resume_reload',lambda:run_model_probe('baseline',smoke_train,smoke_valid,provenance,execution))
 smoke_results['canonical']=run_probe('canonical_cuda_training_checkpoint_resume_reload',lambda:run_model_probe('canonical',smoke_train,smoke_valid,provenance,execution))
 run_probe('checkpoint_integrity',lambda:all(smoke_results[k]['checkpoint_integrity'] for k in ('baseline','canonical')))
 run_probe('offline_checkpoint_reload_inference',lambda:all(smoke_results[k]['offline_reload_inference'] for k in ('baseline','canonical')))
 run_probe('epoch_boundary_recovery_resume',lambda:all(smoke_results[k]['resume_recovery'] for k in ('baseline','canonical')))
 complete_probe_run(REPORT_PATH,VALIDATION_RUN_ID)
 require_full_training_gate(profile,REPORT_PATH,VALIDATION_RUN_ID,context)
 runtime_report['validation_report_status']='validation_passed'; runtime_report['candidate_probe_gate']='passed; profile remains candidate pending review'
 print(json.dumps(smoke_results,indent=2,default=str))
else: print('Skipped by default; set RUN_CUDA_SMOKE_TESTS=True after checks.')

## 8. GPU memory and timing diagnostics
The smoke cell records elapsed wall time and peak allocated memory. These are engineering diagnostics, not benchmark results. On OOM, stop the cell, retain the last completed snapshot, restart from it, and reduce microbatch from 2 to 1 while increasing accumulation from 8 to 16 for both models. Keep effective batch size 16. Do not silently truncate inputs.

In [ ]:
if 'smoke_results' in globals():
 for name,r in smoke_results.items(): print(name,{k:r[k] for k in ('elapsed_seconds','peak_cuda_bytes','effective_precision')})
else: print('No timing sample has run.')

## 9. Baseline training — explicit opt-in
Uses `experiment.train_sst2`; training is never started automatically. Keep the complete output directory to resume from `checkpoints/latest_checkpoint.json` or the checkpoint root.

In [ ]:
BASELINE_OUTPUT=ARTIFACT_ROOT/'baseline'
def train_command(kind,out):
 command=[sys.executable,'-m','experiment.train_sst2','--model-kind',kind,'--split-dir',str(SPLIT_DIR),'--output-dir',str(out),'--model-id',MODEL_ID,'--model-revision',MODEL_REVISION,'--tokenizer-id',TOKENIZER_ID,'--tokenizer-revision',TOKENIZER_REVISION,'--max-expanded-length',str(MAX_EXPANDED_LENGTH),'--max-byte-span-length',str(MAX_BYTE_SPAN_LENGTH),'--batch-size',str(MICROBATCH_SIZE),'--gradient-accumulation-steps',str(GRADIENT_ACCUMULATION_STEPS),'--seed',str(SEED),'--epochs',str(EPOCHS),'--learning-rate',str(LEARNING_RATE),'--weight-decay',str(WEIGHT_DECAY),'--warmup-ratio',str(WARMUP_RATIO),'--gradient-clip-norm',str(GRADIENT_CLIP_NORM),'--mixed-precision',REQUESTED_MIXED_PRECISION,'--validation-interval-steps',str(VALIDATION_INTERVAL_STEPS),'--num-workers',str(NUM_WORKERS),'--device',DEVICE_NAME]
 if ALLOW_PINNED_ASSET_DOWNLOAD: command.append('--allow-hub-download')
 return command
def run_training(kind,out):
 require_current_candidate_gate()
 ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True); log_dir=ARTIFACT_ROOT/'logs'; log_dir.mkdir(exist_ok=True)
 log_path=log_dir/f'{kind}.log'
 with log_path.open('w',encoding='utf-8') as stream: subprocess.run(train_command(kind,out),cwd=REPO_ROOT,check=True,stdout=stream,stderr=subprocess.STDOUT,text=True)
 print('Training log:',log_path)
if RUN_FULL_BASELINE_TRAINING:
 require_current_candidate_gate()
 ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)
 if BASELINE_OUTPUT.exists() and any(BASELINE_OUTPUT.iterdir()): raise FileExistsError('Choose a fresh output directory or explicit resume operation.')
 run_training('baseline',BASELINE_OUTPUT)
else: print('Full baseline training off; set RUN_FULL_BASELINE_TRAINING=True explicitly.')

## 10. Canonical training — explicit opt-in
Uses the same shared settings; architecture-specific differences remain. The comparison is whole-system only, not a byte-feature ablation.

In [ ]:
CANONICAL_OUTPUT=ARTIFACT_ROOT/'canonical'
if RUN_FULL_CANONICAL_TRAINING:
 require_current_candidate_gate()
 ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)
 if CANONICAL_OUTPUT.exists() and any(CANONICAL_OUTPUT.iterdir()): raise FileExistsError('Choose a fresh output directory or explicit resume operation.')
 run_training('canonical',CANONICAL_OUTPUT)
else: print('Full canonical training off; set RUN_FULL_CANONICAL_TRAINING=True explicitly.')

## 11. Validation-only comparison manifest
The existing utility verifies shared settings and declares the comparison whole-system. No held-out split is read.

In [ ]:
from src.comparison import validate_comparison_checkpoints
from src.evaluation import evaluate_checkpoint_split
for kind,out in (('baseline',BASELINE_OUTPUT),('canonical',CANONICAL_OUTPUT)):
 pointer=out/'best_checkpoint.json'
 if pointer.is_file():
  info=json.loads(pointer.read_text()); checkpoint=out/info['checkpoint_dir']
  validation=evaluate_checkpoint_split(checkpoint,SPLIT_DIR,'validation',out/'validation_evaluation',device_name=DEVICE_NAME)
  print(kind+' validation:',json.dumps(validation,indent=2))
if (BASELINE_OUTPUT/'best_checkpoint.json').is_file() and (CANONICAL_OUTPUT/'best_checkpoint.json').is_file():
 bp=json.loads((BASELINE_OUTPUT/'best_checkpoint.json').read_text()); cp=json.loads((CANONICAL_OUTPUT/'best_checkpoint.json').read_text())
 report=validate_comparison_checkpoints(BASELINE_OUTPUT/bp['checkpoint_dir'],CANONICAL_OUTPUT/cp['checkpoint_dir'],SPLIT_DIR)
 print(json.dumps(report,indent=2,default=str)); assert report['comparison_type']=='whole-system; byte-feature effect is not isolated'
else: print('Comparison not run: both full training runs are required.')

## 12. Artifact export and recovery
Export full run bundles (weights, metadata, tokenizer assets, integrity data, resumable snapshots), configs, logs/history and provenance. Do not include source CSVs, credentials, caches or virtual environments. Kaggle working storage is not guaranteed to persist: download the archive or copy it to a user-managed store and verify it before ending the session.

In [ ]:
import shutil
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)
runtime_report['validation_report']=json.loads(REPORT_PATH.read_text(encoding='utf-8')) if REPORT_PATH.exists() else {'report_status':'missing','runs':[]}
runtime_report['validation_report_status']=runtime_report['validation_report'].get('report_status','unknown')
runtime_report['validation_context_fingerprint']=runtime_report['validation_report'].get('runs',[])[-1].get('context_fingerprint') if runtime_report['validation_report'].get('runs') else None
(ARTIFACT_ROOT/'runtime_report.json').write_text(json.dumps(runtime_report,indent=2,default=str),encoding='utf-8')
(ARTIFACT_ROOT/'workflow_config.json').write_text(json.dumps({'shared':SHARED,'requested_precision':REQUESTED_MIXED_PRECISION,'effective_precision_probe':effective_precision,'model_revision':MODEL_REVISION,'tokenizer_revision':TOKENIZER_REVISION,'verified_hf_assets':asset_verification,'comparison':'whole-system only; byte-feature effect is not isolated'},indent=2),encoding='utf-8')
shutil.copy2(SPLIT_DIR/'manifest.json',ARTIFACT_ROOT/'dataset_manifest.json')
archive=shutil.make_archive(str(Path('/kaggle/working')/f'nlp-defense-m24-export-{SESSION_ID}'),'zip',root_dir=ARTIFACT_ROOT)
print({'archive':archive,'bytes':Path(archive).stat().st_size,'run_dirs':[p.name for p in ARTIFACT_ROOT.iterdir() if p.is_dir()],'resume':'Restore complete run directory; resume using checkpoints/latest_checkpoint.json or the checkpoint root.','persistence':'Download/export ZIP or copy to a user-managed store; Kaggle storage may be ephemeral.'})